# AI Agent Tool Selection: local LLM vs trained smart tool

A single, self-contained comparison that a reviewer can run from the CSV alone.

For a set of tasks we compare two ways of choosing the cybersecurity tool:
1. **Local LLM (Llama 3.2 3B via Ollama)** — the choice it made during data
   collection, recorded in the `tool_predicted` column.
2. **Smart tool (trained SVM)** — a TF-IDF + linear SVM trained on the labeled
   prompts.

Both are scored against the correct tool (`tool_ground_truth`). No SSH, no live
model, and no network are needed to reproduce this: it runs from `cyber_logs.csv`.
(An optional last cell re-runs the LLM live if you have Ollama + llama3.2.)

**Run:** `pip install pandas scikit-learn` then Restart & Run All.

In [1]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

for p in ["cyber_logs.csv", os.path.join("data","trajectories","cyber_logs.csv")]:
    if os.path.exists(p):
        CSV = p; break
df = pd.read_csv(CSV)
print("Loaded", CSV, "->", len(df), "rows")

Loaded cyber_logs.csv -> 640 rows


## 1. Train the smart tool (SVM)

80/20 split; TF-IDF over the prompt; linear SVM. The 10 comparison tasks are
drawn from the held-out test set, so the SVM has not seen them.

In [2]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["tool_ground_truth"])

vec = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
svm = SVC(kernel="linear", random_state=42).fit(
    vec.fit_transform(train_df["prompt"]), train_df["tool_ground_truth"])
print("SVM trained on", len(train_df), "examples; test set has", len(test_df))

SVM trained on 512 examples; test set has 128


## 2. The 10 tasks: LLM choice vs smart-tool choice

In [3]:
tasks = test_df.sample(10, random_state=1).copy()
tasks["llm_pick"] = tasks["tool_predicted"]
tasks["smart_tool_pick"] = svm.predict(vec.transform(tasks["prompt"]))

view = tasks[["prompt", "tool_ground_truth", "llm_pick", "smart_tool_pick"]].reset_index(drop=True)
pd.set_option("display.max_colwidth", 60)
view

,prompt,tool_ground_truth,llm_pick,smart_tool_pick
0,Start a remote shell on the defender via SSH,SSHConnect,SSHConnect,SSHConnect
1,Run an nmap -sV version scan of the target,NmapScan,NmapScan,NmapScan
2,Report known security advisories for MySQL 1.18,CheckVulnerability,CheckVulnerability,CheckVulnerability
3,Fingerprint the service versions on the defender with nmap,NmapScan,NmapScan,NmapScan
4,Count the failed login attempts right now on this box,CheckFailedLogins,CheckFailedLogins,CheckFailedLogins
5,Do a service and version detection scan on the target VM,NmapScan,NmapScan,NmapScan
6,Show me the recent general system log entries just now o...,ReadSyslog,ReadSyslog,ReadSyslog
7,Look up known CVEs for vsftpd 7.2,CheckVulnerability,CheckVulnerability,CheckVulnerability
8,"No need for service versions, only tell me which ports a...",PortScan,NmapScan,PortScan
9,Report known security advisories for http 8.0.32,CheckVulnerability,CheckVulnerability,CheckVulnerability


## 3. Score

In [4]:
llm_correct = (tasks["llm_pick"] == tasks["tool_ground_truth"]).sum()
svm_correct = (tasks["smart_tool_pick"] == tasks["tool_ground_truth"]).sum()
print(f"Local LLM (Llama 3.2) correct : {llm_correct}/10")
print(f"Smart tool (SVM)      correct : {svm_correct}/10")

Local LLM (Llama 3.2) correct : 9/10
Smart tool (SVM)      correct : 10/10


## 4. (Optional) re-run the LLM live

Only if you have Ollama with `llama3.2:3b` pulled. This regenerates the LLM's
tool choices for the same 10 prompts; skip it otherwise.

In [5]:
try:
    import ollama
    TOOLS = sorted(df["tool_ground_truth"].unique())
    sys = ("You are a tool selector. Reply with exactly one word, the tool to use, "
           "chosen from: " + ", ".join(TOOLS) + ".")
    live = []
    for p in tasks["prompt"]:
        r = ollama.chat(model="llama3.2:3b",
                        messages=[{"role": "system", "content": sys},
                                  {"role": "user", "content": p}],
                        options={"temperature": 0, "seed": 42})
        raw = r["message"]["content"].strip()
        pick = next((t for t in TOOLS if t in raw), "Unknown")
        live.append(pick)
    tasks["llm_live"] = live
    print("live LLM correct:", (tasks["llm_live"] == tasks["tool_ground_truth"]).sum(), "/10")
    print(tasks[["tool_ground_truth", "llm_pick", "llm_live", "smart_tool_pick"]].to_string())
except Exception as e:
    print("Skipped live LLM run (Ollama not available):", e)

Skipped live LLM run (Ollama not available): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


## Takeaway

On held-out tasks the trained smart tool matches or beats the local LLM at
choosing the right cybersecurity tool, while running locally with no model call.
The LLM's recorded choices are in `tool_predicted`; the optional cell shows they
can be regenerated with Ollama.